# Make 2D topo and dtopo

In [ ]:
%matplotlib inline

In [ ]:
from pylab import *
from clawpack.geoclaw import topotools, kmltools, dtopotools
from clawpack.visclaw import gridtools, plottools
from clawpack.geoclaw.util import haversine, gctransect
from clawpack.geoclaw.data import Rearth
#from clawpack.clawutil.util import fullpath_import
from scipy.interpolate import interp1d
from scipy.interpolate import RegularGridInterpolator, griddata
import os,sys
import mapper

In [ ]:
save_figs = True

In [ ]:
fname1d = '1d_radial/celledges_1m.txt'
rdeg, z = loadtxt(fname1d, skiprows=1, unpack=True)

In [ ]:
figure(figsize=(11,6))
plot(rdeg,z,'g')
grid(True)
xlabel('polar angle')
ylabel('elevation (m)')
title('Topography transect');

In [ ]:
deg2m = Rearth*pi/180   # approx 111e3 meters per degree latitude

# extend grid to go from 0 to 50 degrees and then convert to meters:
rm_ext = hstack((0, rdeg, 50)) * deg2m
z_ext = hstack((z[0],z,z[-1]))

figure(figsize=(11,6))

plot(rm_ext/1e3,z_ext,'g')
grid(True)
xlabel('distance from pole (km)')
title('Topography')
xlim(2200,2550)

if save_figs:
    fname = 'topo1.png'
    savefig(fname, bbox_inches='tight')
    print('Created ',fname)

figure(figsize=(11,6))

plot(rm_ext/1e3,z_ext,'g')
grid(True)
xlabel('distance from pole (km)')
title('Nearshore Topography')
xlim(2515,2522)
ylim(-20,20)

if save_figs:
    fname = 'topo2.png'
    savefig(fname, bbox_inches='tight')
    print('Created ',fname)

In [ ]:
rm_fcn = interp1d(rm_ext, z_ext, kind='linear')

In [ ]:
jshore = where(z_ext>0)[0].min()
rm_shore = rm_ext[jshore]
print(f'Distance from center to shore is {rm_shore/1e3:.3f} km')

In [ ]:
if 0:
    theta = 20 *pi/180  # angle of transect for checking solution
    r_shore = 28.3665
    rm_shore = r_shore*deg2m
    #x_shore = r_shore * cos(theta)
    #y_shore = r_shore * sin(theta)
    #print(f'x_shore = {x_shore:.5f}, y_shore = {y_shore:.5f}')


In [ ]:
 if 0:
    r1 = 0
    r2 = 30
    x1 = r1 * cos(theta)
    x2 = r2 * cos(theta)
    y1 = r1 * sin(theta)
    y2 = r2 * sin(theta)
    
    # define 10' grid
    dx_10m = 10/60.
    x_10m = arange(x1, x2+dx_10m, dx_10m)
    y_10m = arange(y1, y2+dx_10m, dx_10m)
    
    X_10m, Y_10m = meshgrid(x_10m, y_10m)
    R_10m = haversine(X_10m, Y_10m, 0, 0)  # great circle distance from each point on grid to pole
    Z_10m = rm_fcn(R_10m)

    r1 = 0
    r2 = 30
    x1 = r1 * cos(theta)
    x2 = r2 * cos(theta)
    y1 = r1 * sin(theta)
    y2 = r2 * sin(theta)
    
    # define 15" grid
    dx_15s = 15/3600.
    x_15s = arange(x1, x2+dx_15s, dx_15s)
    y_15s = arange(y1, y2+dx_15s, dx_15s)
    
    X_15s, Y_15s = meshgrid(x_15s, y_15s)
    R_15s = haversine(X_15s, Y_15s, 0, 0)  # great circle distance from each point on grid to pole
    Z_15s = rm_fcn(R_15s)

    contour(X_15s, Y_15s, Z_15s)
    axis('scaled')
    #plot([r_shore],[0],'bo')
    #plot([x_shore],[y_shore],'ro')

## Rotate axis of symmetry from pole to latitude y0

In [ ]:
x0 = 50 # rotate down at this longitude
y0 = 40 # rotate down to this latitude

x1 = x0 - 40
x2 = x0 + 40
y1 = y0 - 30
y2 = y0 + 30

# define 10' grid for plotting radial ocean
dx_10m = 10/60.
x_10m = arange(x1, x2+dx_10m, dx_10m)
y_10m = arange(y1, y2+dx_10m, dx_10m)

X_10m, Y_10m = meshgrid(x_10m, y_10m)
R_10m = haversine(X_10m, Y_10m, x0, y0)  # great circle distance from each point on grid to rotated axis
Z_10m = rm_fcn(R_10m)


In [ ]:
if 0:
    xtrans, ytrans = gctransect(x0,y0, 35,65, 1000, coords='E')

    contour(X_10m, Y_10m, Z_10m)
    axis('scaled')
    plot(xtrans, ytrans, 'r')
    text(-3,43,'great circle transect',ha='left',rotation=40,color='r')
    title('Circular ocean with contours of topography');


In [ ]:

clines = [-2000, -1000]
contour(X_10m, Y_10m, Z_10m, clines, colors='g', linestyles='-')
contour(X_10m, Y_10m, Z_10m, [0], colors='b') # shoreline
axis('scaled')
d = rm_shore  # distance to shore
phi = y0 # shift down from pole
for theta in arange(5, 360, 10):
    xhat, yhat = mapper.latlong(d,theta,phi,Rearth)
    xhat += x0
    xtrans, ytrans = gctransect(x0,y0, xhat, yhat, 1000, coords='E')
    plot(xtrans, ytrans, 'r')
xlabel('longitude')
ylabel('latitude')
title('Contours of 2D topography and great circle transects\n' \
     +f'With axis of symmetry passing through {x0,y0}');

if save_figs:
    fname = 'topo3.png'
    savefig(fname, bbox_inches='tight')
    print('Created ',fname)

## Check

In [ ]:
if 0:
    topo_10m = topotools.Topography()
    topo_10m.set_xyZ(x_10m, y_10m, Z_10m)
    topo_10m_fcn = topo_10m.make_function()
    ztrans = topo_10m_fcn(xtrans, ytrans)
    dtrans = haversine(xtrans, ytrans, x0, y0)
    
    figure(figsize=(10,6))
    plot(rm_ext/1e3, z_ext, 'r', label='1D radial topo on fine grid')
    plot(dtrans/1e3, ztrans,'b', label='transect of 2D topo')
    xlim(2200, 3200)
    grid(True)
    title('Transect of 10 arcminute 2D topo');
    xlabel('distance from ocean center (km)');
    legend(loc='upper left', framealpha=1)

## Choose one transect, computational domain

The 2D simulation will be over a small coastal region from the circular ocean defined above.  Choose a particular radial transect to focus on and the coastal domain along this transect that covers the continental shelf and onshore region.



In [ ]:
fig,ax = subplots(figsize=(6,8))

clines = [-2000, -1000]
contour(X_10m, Y_10m, Z_10m, clines, colors='g', linestyles='-')
contour(X_10m, Y_10m, Z_10m, [0], colors='b') # shoreline

ax.set_aspect(1/cos(60*pi/180))  # correct near desired coastal location
theta = 210 # choose transect at this angle
d = rm_shore + 3e3  # extend far enough past shore to capture inundation region
xhat, yhat = mapper.latlong(d,theta,y0,Rearth)
xhat += x0

# initial transect from ocean center
xtrans0, ytrans0 = gctransect(x0,y0, xhat, yhat, 5000, coords='E')

plot(xtrans0, ytrans0, 'r')

# plot computational domain to use as blue box:
x1 = 60
x2 = 80
y1 = 50
y2 = 65

plottools.plotbox([x1,x2,y1,y2])
grid(True);
xlim(35,85)
ylim(35,70)

title('Contours of 2D topography and great circle transect\nBlue box = computational domain');

if save_figs:
    fname = 'topo4.png'
    savefig(fname, bbox_inches='tight')
    print('Created ',fname)

In [ ]:
# define 15" grid over domian:

dx_15s = 15/3600.
x_15s = arange(x1, x2+dx_15s, dx_15s)
y_15s = arange(y1, y2+dx_15s, dx_15s)

X_15s, Y_15s = meshgrid(x_15s, y_15s)
R_15s = haversine(X_15s, Y_15s, x0, y0)  # great circle distance from each point on grid to pole
Z_15s = rm_fcn(R_15s)

topo_15s = topotools.Topography()
topo_15s.set_xyZ(x_15s, y_15s, Z_15s)
topo_15s_fcn = topo_15s.make_function()

print(f'Defined topo_15s on 15" grid with {len(x_15s)} by {len(y_15s)} grid points')


In [ ]:
# transect through the domain at 500 m spacing, roughly 15" resolution:
j1 = where(xtrans0 < x1)[0].max()
length = haversine(xtrans0[j1], ytrans0[j1], xtrans0[-1], ytrans0[-1])
npts = int(round(length / 500.))
xtrans_15s, ytrans_15s = gctransect(xtrans0[j1], ytrans0[j1], 
                                    xtrans0[-1], ytrans0[-1], npts, coords='E')

# topography on this transect:
ztrans_15s = topo_15s_fcn(xtrans_15s, ytrans_15s)
# distance from ocean center:
dtrans_15s = haversine(xtrans_15s, ytrans_15s, x0, y0)

### Plot topo interpolated from grid with original 1D topo

As a sanity check...

In [ ]:
figure(figsize=(10,6))
plot(rm_ext/1e3, z_ext, 'r', label='1D radial topo on fine grid')
plot(dtrans_15s/1e3, ztrans_15s,'b', label='transect of 2D topo')
xlim(2000, 2600)
grid(True)
title('Transect of 15 arcsecond 2D topo');
xlabel('distance from ocean center (km)');
legend(loc='upper left', framealpha=1)

### Plot the 2D topography at this resolution

In [ ]:
contour(X_15s, Y_15s, Z_15s)
gca().set_aspect(1/cos(60*pi/180))
plot(xtrans_15s, ytrans_15s, 'r')
#text(-3,43,'great circle transect',ha='left',rotation=40,color='r')
#axis([x1,x2,y1,y2])
title('Computation domain and transect');

topo_15s.plot(limits=(-2000,2000))
title('Topography');

## Choose 3 arcsecond grid

This grid covers the continental shelf in the region around the selected transect.

In [ ]:
x1_3s = 68
x2_3s = 72
y1_3s = 56
y2_3s = 59

# transect through this region at 90 m spacing, roughly 3" resolution:
j1 = where(xtrans0 < x1_3s)[0].max()
length = haversine(xtrans0[j1], ytrans0[j1], xtrans0[-1], ytrans0[-1])
npts = int(round(length / 90.))
xtrans_3s, ytrans_3s = gctransect(xtrans0[j1], ytrans0[j1], 
                                    xtrans0[-1], ytrans0[-1], npts, coords='E')


contour(X_15s, Y_15s, Z_15s)
gca().set_aspect(1/cos(60*pi/180))

plot(xtrans_3s, ytrans_3s, 'r')
title('3 arcsecond region and transect');

plottools.plotbox([x1_3s,x2_3s,y1_3s,y2_3s])
grid(True);

In [ ]:
# define 3" grid
dx_3s = 3/3600.
x_3s = arange(x1_3s, x2_3s+dx_3s, dx_3s)
y_3s = arange(y1_3s, y2_3s+dx_3s, dx_3s)

X_3s, Y_3s = meshgrid(x_3s, y_3s)
R_3s = haversine(X_3s, Y_3s, x0, y0)  # great circle distance from each point on grid to pole
Z_3s = rm_fcn(R_3s)

topo_3s = topotools.Topography()
topo_3s.set_xyZ(x_3s, y_3s, Z_3s)
topo_3s_fcn = topo_3s.make_function()

ztrans_3s = topo_3s_fcn(xtrans_3s, ytrans_3s)
dtrans_3s = haversine(xtrans_3s, ytrans_3s, x0, y0)

In [ ]:
#contour(X_3s, Y_3s, Z_3s)
#axis('scaled')
topo_3s.plot(limits=(-200,200))
gca().set_aspect(1/cos(60*pi/180))

plot(xtrans_3s, ytrans_3s, 'r')
title('3 second grid');

In [ ]:
figure(figsize=(10,6))
plot(rm_ext/1e3, z_ext, 'r', label='1D radial topo on fine grid')
plot(dtrans_3s/1e3, ztrans_3s,'b', label='transect of 2D topo')
xlim(2000, 2530)
ylim(-3000,200)
grid(True)
title('Transect of 3 arcsecond 2D topo');
xlabel('distance from ocean center (km)');
legend(loc='upper left', framealpha=1);

In [ ]:
if 0:
    j1 = where(dtrans<3025e3)[0].max()
    j2 = where(dtrans>3151e3)[0].min()
    #j, dtrans[j], dtrans[j+1], len(dtrans)
    dd = dtrans[j2] - dtrans[j1]
    nn = int(float(dd/100))
    print('nn = ',nn)
    xtrans1,ytrans1 = gctransect(xtrans[j1],ytrans[j1], xtrans[j2],ytrans[j2],nn)
    ztrans1 = topo_3s_fcn(xtrans1, ytrans1)
    dtrans1 = haversine(xtrans1, ytrans1, x0, y0)
    
    figure(figsize=(10,6))
    plot(rm_ext/1e3, z_ext, 'r', label='1D radial topo on fine grid')
    plot(dtrans1/1e3, ztrans1,'b', label='transect of 2D topo')
    xlim(3000, 3200)
    grid(True)
    title('Transect of 3 arcsecond 2D topo');
    xlabel('distance from ocean center (km)');
    legend(loc='upper left', framealpha=1)

## Choose 1/3 arcsecond grid

In [ ]:
x1_3s, y1_3s, x2_3s, y2_3s

In [ ]:
x1_13s = 71.2
x2_13s = 71.45
y1_13s = 58
y2_13s = 58.15

# transect through this region at 10 m spacing, roughly 1/3" resolution:
j1 = where(xtrans0 < x1_13s)[0].max()
length = haversine(xtrans0[j1], ytrans0[j1], xtrans0[-1], ytrans0[-1])
npts = int(round(length / 10.))
xtrans_13s, ytrans_13s = gctransect(xtrans0[j1], ytrans0[j1], 
                                    xtrans0[-1], ytrans0[-1], npts, coords='E')

contour(X_3s, Y_3s, Z_3s)
gca().set_aspect(1/cos(60*pi/180))

plot(xtrans_13s, ytrans_13s, 'r')
title('1/3 arcsec grid within 3 arcsec');

plottools.plotbox([x1_13s,x2_13s,y1_13s,y2_13s])

#axis([78,82,60,63])
axis([x1_3s, x2_3s, y1_3s, y2_3s])

grid(True);


In [ ]:
# define 1/3" grid
dx_13s = 1/(3*3600.)
x_13s = arange(x1_13s, x2_13s+dx_13s, dx_13s)
y_13s = arange(y1_13s, y2_13s+dx_13s, dx_13s)

X_13s, Y_13s = meshgrid(x_13s, y_13s)
R_13s = haversine(X_13s, Y_13s, x0, y0)  # great circle distance from each point on grid to pole
Z_13s = rm_fcn(R_13s)

topo_13s = topotools.Topography()
topo_13s.set_xyZ(x_13s, y_13s, Z_13s)
topo_13s_fcn = topo_13s.make_function()
ztrans_13s = topo_13s_fcn(xtrans_13s, ytrans_13s)
dtrans_13s = haversine(xtrans_13s, ytrans_13s, x0, y0)

In [ ]:
Z_13s.shape

In [ ]:
rm_shore/1e3, rm_ext[-1]/1e3, dtrans_13s[-1]/1e3, xtrans0[-1], xtrans_13s[-1], x2_13s

In [ ]:
figure(figsize=(10,6))
plot(rm_ext/1e3, z_ext, 'r', label='1D radial topo on fine grid')
plot(dtrans_13s/1e3, ztrans_13s,'b', label='transect of 2D topo')
#xlim(2518, 2521)
xlim(rm_shore/1e3 - 2, rm_shore/1e3 + 3)
ylim(-15,15)
grid(True)
title('Transect of 3 arcsecond 2D topo');
xlabel('distance from ocean center (km)');
legend(loc='upper left', framealpha=1);

In [ ]:
if 0:
    j1 = where(dtrans<3145e3)[0].max()
    j2 = where(dtrans>3151e3)[0].min()
    #j, dtrans[j], dtrans[j+1], len(dtrans)
    dd = dtrans[j2] - dtrans[j1]
    nn = int(float(dd/5))
    print('nn = ',nn)
    xtrans1,ytrans1 = gctransect(xtrans[j1],ytrans[j1], xtrans[j2],ytrans[j2],nn)
    ztrans1 = topo_13s_fcn(xtrans1, ytrans1)
    dtrans1 = haversine(xtrans1, ytrans1, x0, y0)
    
    figure(figsize=(10,6))
    plot(rm_ext/1e3, z_ext, 'r', label='1D radial topo on fine grid')
    plot(dtrans1/1e3, ztrans1,'b', label='transect of 2D topo')
    xlim(3145, 3152)
    ylim(-20,20)
    grid(True)
    title('Transect of 1/3 arcsecond 2D topo');
    xlabel('distance from ocean center (km)');
    legend(loc='upper left', framealpha=1)

## Write GeoClaw topo files:

In [ ]:
if 1:
    fname = 'topo15s.asc'
    topo_15s.write(fname, topo_type=3, Z_format=' %.3f', header_style='asc')
    print(f'Created {fname} with shape {topo_15s.Z.shape}')
    
    fname = 'topo3s.asc'
    topo_3s.write(fname, topo_type=3, Z_format=' %.3f', header_style='asc')
    print(f'Created {fname} with shape {topo_3s.Z.shape}')
    
    fname = 'topo13s.asc'
    topo_13s.write(fname, topo_type=3, Z_format=' %.3f', header_style='asc')
    print(f'Created {fname} with shape {topo_13s.Z.shape}')


## Examine onshore topography

In [ ]:
print(f'Shore is at {rm_shore/1e3:.3f} km from axis')
xs,ys = mapper.latlong(rm_shore, 210, 40., Rearth)
xs += x0
print(f'On this transect, shore is at ({xs:.5f}, {ys:.5f})')

topo_13s.plot(limits=(-20,20))
plot(xtrans_13s, ytrans_13s, 'r')
plot([xs],[ys],'ro')  # shore location
title('Shore topography with shoreline at red point');

# plot with gauges
d_shore = 2518.589 * 1e3
d_gauges = np.arange(d_shore-400, d_shore+2501, 200)
gauges = []
for k,d in enumerate(d_gauges):
    gaugeno = k+1
    x,y = mapper.latlong(d, 210, 40., Rearth)
    x = x + 50
    gauges.append((x,y))
gauges = array(gauges)
topo_13s.plot(limits=(-20,20))
plot(xtrans_13s, ytrans_13s, 'r')
plot(gauges[:,0],gauges[:,1],'ro')
xlim(71.35, 71.42)
ylim(58.06,58.1)
title('With gauge locations');

In [ ]:
js = where(dtrans_13s < rm_shore)[0].max()
xtrans_shore = xtrans_13s[js-20:]
ytrans_shore = ytrans_13s[js-20:]
dtrans_shore = dtrans_13s[js-20:]
print(f'{len(xtrans_shore)} points in transect_shore')

In [ ]:
fname = 'transect_shore.txt'
savetxt(fname, vstack((xtrans_shore, ytrans_shore, dtrans_shore)).T, fmt='%.7f')
print('Created ',fname)

In [ ]:
xtrans_shore[0], xtrans_shore[-1], ytrans_shore[0], ytrans_shore[-1]

## Make 2d dtopo

In [ ]:
dtopo_name = 'gaussian5m'
dtopo_1d = loadtxt(f'1d_radial/{dtopo_name}_dtopo_1m.dtt1')
r1d = dtopo_1d[:,1] * deg2m
dz1d = dtopo_1d[:,2]
dz_fcn = interp1d(r1d, dz1d, kind='linear', fill_value=0., bounds_error=False)

In [ ]:
dZ_15s = dz_fcn(R_15s)  # evaluation on 2D 15" grid covering domain
dtopo_2d = dtopotools.DTopography()
dtopo_2d.X = X_15s
dtopo_2d.Y = Y_15s
dtopo_2d.dZ = array(dZ_15s, ndmin=3)
dtopo_2d.times = [1.]  # time of instantaneous displacement

In [ ]:
fig,ax = subplots(figsize=(6,8))
dtopo_2d.plot_dZ_colors(t=2, axes=ax, dZ_interval=1e6)
ax.set_aspect(1/cos(60*pi/180))
contour(X_15s, Y_15s, Z_15s, [-2000, -1000, 0], colors='g')
plot(xtrans_15s, ytrans_15s, 'k', label='great circle transect')
legend()
ax.set_title('Colors show seafloor deformation\ntopo contours in green');

In [ ]:
fname = f'{dtopo_name}_dtopo_15s.dtt3'
dtopo_2d.write(fname, dtopo_type=3)
print('Created ',fname)